# Downloading the DataSet

Config - loads project paths once. Works from any machine/clone location; only `config.py` needs to exist at the repo root.

In [2]:
# Cell 1 — Setup
from config import BASE_DIR, DATASETS_DIR
from kaggle.api.kaggle_api_extended import KaggleApi
import random, os, time
from tqdm import tqdm

api = KaggleApi()
api.authenticate()
dataset = "xdxd003/ff-c23"
print("Kaggle authenticated. Target dataset:", dataset)

# List all files in the dataset (this may take a moment)
files = api.dataset_list_files(dataset).files

print(f"Total files in dataset: {len(files)}")

Kaggle authenticated. Target dataset: xdxd003/ff-c23
Total files in dataset: 20


In [3]:
# Cell 2 — List all files (paginated)
all_files = []
page_token = None
while True:
    result = api.dataset_list_files(dataset, page_token=page_token, page_size=500)
    all_files.extend(result.files)
    page_token = result.next_page_token
    if not page_token:
        break

print(f"Total files collected: {len(all_files)}")

Total files collected: 7010


In [4]:
# Look at the first file object's actual attributes
print(dir(files[0]))

['__class__', '__contains__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_columns', '_creation_date', '_dataset_ref', '_description', '_fields', '_file_type', '_freeze', '_get_field', '_is_frozen', '_name', '_owner_ref', '_ref', '_total_bytes', '_url', 'body_fields', 'columns', 'creation_date', 'dataset_ref', 'description', 'endpoint', 'endpoint_path', 'file_type', 'from_dict', 'from_json', 'method', 'name', 'owner_ref', 'prepare_from', 'ref', 'to_dict', 'to_field_map', 'to_json', 'total_bytes', 'url']


In [5]:
for f in files:
    print(f.name, f.total_bytes)

FaceForensics++_C23/DeepFakeDetection/01_02__meeting_serious__YVGY8LOK.mp4 6745903
FaceForensics++_C23/DeepFakeDetection/01_02__outside_talking_still_laughing__YVGY8LOK.mp4 5290755
FaceForensics++_C23/DeepFakeDetection/01_02__talking_against_wall__YVGY8LOK.mp4 3471524
FaceForensics++_C23/DeepFakeDetection/01_02__walk_down_hall_angry__YVGY8LOK.mp4 1164071
FaceForensics++_C23/DeepFakeDetection/01_02__walking_down_indoor_hall_disgust__YVGY8LOK.mp4 12640206
FaceForensics++_C23/DeepFakeDetection/01_03__hugging_happy__ISF9SP4G.mp4 9123749
FaceForensics++_C23/DeepFakeDetection/01_03__kitchen_pan__JZUXXFRB.mp4 3485507
FaceForensics++_C23/DeepFakeDetection/01_03__podium_speech_happy__480LQD1C.mp4 5056283
FaceForensics++_C23/DeepFakeDetection/01_03__talking_against_wall__JZUXXFRB.mp4 3489591
FaceForensics++_C23/DeepFakeDetection/01_04__hugging_happy__GBC7ZGDP.mp4 8246897
FaceForensics++_C23/DeepFakeDetection/01_04__meeting_serious__0XUW13RW.mp4 6563740
FaceForensics++_C23/DeepFakeDetection/01_04

In [6]:
help(api.dataset_list_files)

Help on method dataset_list_files in module kaggle.api.kaggle_api_extended:

dataset_list_files(dataset, page_token=None, page_size=20) method of kaggle.api.kaggle_api_extended.KaggleApi instance
    Lists files for a dataset.
    
    Args:
        dataset: The string identifier of the dataset, in the format [owner]/[dataset-name].
        page_token: The page token for pagination.
        page_size: The number of items per page.



In [7]:
# Cell 3 — Filter and sample
original_files = [f.name for f in all_files if f.name.startswith("FaceForensics++_C23/original/")]
deepfakes_files = [f.name for f in all_files if f.name.startswith("FaceForensics++_C23/Deepfakes/")]

print(f"Original (real) videos found: {len(original_files)}")
print(f"Deepfakes videos found: {len(deepfakes_files)}")

random.seed(42)
SAMPLE_SIZE = 1000
sampled_original = random.sample(original_files, min(SAMPLE_SIZE, len(original_files)))
sampled_deepfakes = random.sample(deepfakes_files, min(SAMPLE_SIZE, len(deepfakes_files)))

print(f"Selected {len(sampled_original)} real videos")
print(f"Selected {len(sampled_deepfakes)} fake videos")

Original (real) videos found: 1000
Deepfakes videos found: 1000
Selected 1000 real videos
Selected 1000 fake videos


Real download loop is in the cell given below

In [8]:
# Cell 4 — Resilient, resumable, self-healing download function
def download_with_retry(filenames, dest_folder, max_retries=3, max_passes=3):
    """Downloads until every file succeeds or max_passes is exhausted —
    handles any number of stragglers automatically, not just one."""
    os.makedirs(dest_folder, exist_ok=True)

    for pass_num in range(1, max_passes + 1):
        existing = set(os.listdir(dest_folder))
        remaining = [f for f in filenames if f.split("/")[-1] not in existing]

        if not remaining:
            print(f"All {len(filenames)} files present after pass {pass_num - 1}.")
            return

        print(f"Pass {pass_num}: {len(remaining)} file(s) remaining...")
        for filename in tqdm(remaining):
            attempt = 0
            while attempt < max_retries:
                try:
                    api.dataset_download_file(dataset, file_name=filename, path=dest_folder)
                    break
                except Exception as e:
                    attempt += 1
                    if attempt == max_retries:
                        print(f"  FAILED: {filename} -> {e}")
                    else:
                        time.sleep(3 * attempt)  # backs off a bit more each retry

    final_count = len(os.listdir(dest_folder))
    print(f"Stopped after {max_passes} passes. {final_count}/{len(filenames)} files on disk.")

In [9]:
# Cell 5 — Run the downloads
real_dest = str(DATASETS_DIR / "raw_original")
fake_dest = str(DATASETS_DIR / "raw_deepfakes")

print("Downloading real (original) videos...")
download_with_retry(sampled_original, real_dest)

print("\nDownloading fake (Deepfakes) videos...")
download_with_retry(sampled_deepfakes, fake_dest)

All 1000 files present after pass 0.

All 1000 files present after pass 0.


In [10]:
import os

existing_real = set(os.listdir(real_dest))
print(f"Already downloaded: {len(existing_real)} real videos")

Already downloaded: 1000 real videos


In [11]:
import os

real_count = len(os.listdir(real_dest))
fake_count = len(os.listdir(fake_dest))

print(f"Real videos on disk: {real_count}")
print(f"Fake videos on disk: {fake_count}")

Real videos on disk: 1000
Fake videos on disk: 1000


In [14]:
# Cell 6 — Final verification
real_count = len(os.listdir(real_dest))
fake_count = len(os.listdir(fake_dest))
print(f"Real videos on disk: {real_count} / {len(sampled_original)}")
print(f"Fake videos on disk: {fake_count} / {len(sampled_deepfakes)}")

if real_count == len(sampled_original) and fake_count == len(sampled_deepfakes):
    print("✅ Download complete — all files present.")
else:
    print("⚠️  Some files still missing — re-run Cell 5, it will pick up only what's missing.")

Real videos on disk: 1000 / 1000
Fake videos on disk: 1000 / 1000
✅ Download complete — all files present.
